In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path
import panel as pn
from bokeh.plotting import figure
from bokeh.io import curdoc, output_notebook
from bokeh.models import LinearAxis, Range1d

output_notebook(hide_banner=True)

pn.extension(design="fast", theme="dark")

DAYS_S = 60 * 60 * 24  # how many seconds in a day
NUM_DAYS = 2  # how many days of data to read
FREQ = "15min"  # frequency to group data items by

COLORS = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A", "#19D3F3", "#FF6692", "#B6E880"]

curdoc().theme = "dark_minimal"

db = sorted(Path("..").glob("*.sqlite"))[-1]  # most recent database

with sqlite3.connect(db) as con:
    df = pd.read_sql_query(f"SELECT * from readings order by date desc limit {int(DAYS_S*NUM_DAYS)}", con)

df.date = pd.to_datetime(df.date, format="mixed", errors="coerce")
df.set_index("date", inplace=True)
df.sort_index(inplace=True)
df_sel = df.groupby(pd.Grouper(freq=FREQ)).mean().dropna(how="all")

In [ ]:
trends = []
for c, col in zip(["temperature", "humidity", "pressure", "iaq"], COLORS):
    data = df_sel[c].to_numpy()
    n = pn.indicators.Trend(
        label=c.title(),
        data={"x": df_sel.index, "y": data},
        layout="row",
        width=280,
        height=100,
        plot_type="area",
        plot_color=col,
    )
    trends.append(n)

GAS_COLS = ["gas_resistance", "oxidising", "reducing", "nh3"]

y = df_sel[GAS_COLS]

gasfig = figure(
    height=200,
    x_axis_type="datetime",
    x_axis_location="above",
    title="Gasses",
    tools="reset,xpan,xbox_zoom",
)

gasfig.yaxis.visible = False
gasfig.xaxis.visible = False
gasfig.toolbar.logo = None

for idx, g in enumerate(GAS_COLS):
    color = COLORS[idx % len(COLORS)]
    gasfig.extra_y_ranges[g] = Range1d(y[g].min(), y[g].max())
    ax = LinearAxis(y_range_name=g, axis_label=g.replace("_", " ").title())
    ax.axis_label_text_color = color
    gasfig.add_layout(ax, "left")
    gasfig.line(y.index, y[g], color=color)

RGB_COLS = ["r", "g", "b", "c"]

y = df_sel[RGB_COLS]

rgbfig = figure(
    height=200,
    x_axis_type="datetime",
    x_axis_location="above",
    title="RGBL Values",
    tools="reset,xpan,xbox_zoom",
    x_range=gasfig.x_range,
    y_axis_label="RGBC Values",
)
rgbfig.xaxis.visible = False
rgbfig.toolbar.logo = None

for c, color in zip(RGB_COLS, ["red", "green", "blue", "white"]):
    rgbfig.line(y.index, y[c], color=color, line_width=2)
    rgbfig.varea(y.index, y1=y[c] * 0, y2=y[c], fill_color=color, fill_alpha=0.25)

pn.Column("# Sensor Monitor", pn.Row(trends[0], trends[1]), pn.Row(trends[2], trends[3]), gasfig, rgbfig)